---
title: "Exercise 2"
editor: source
editor_options: 
  chunk_output_type: console
jupyter: ir
---

# Enriched Heatmaps and identification of regions with different activities

## Learning Objectives

- By the end of this section, you will be able to:
- Distinguish between when to use `Heatmap()` and `HeatmapAnnotation()`
- Generate row split annotations for enriched heatmaps
- Add multiple categorical annotations with separate color legends
- Avoid common data handling errors in `heatmap` plotting

## Load Libraries

In [ ]:
knitr::opts_chunk$set(warning = FALSE) 

[Follow the instructions](https://github.com/mskilab-org/gUtils) to install `gUtils`.

In [ ]:
#| warning: false
library(SummarizedExperiment)
library(EnrichedHeatmap)
library(gUtils)
library(rtracklayer)
library(circlize)
library(GenomicRanges)

## ATAC SE object
Here we load the `SummarizedExperiment` for the ATAC-seq data

In [ ]:
atac <- readRDS("data/atac_se.rds")

## ATAC peaks info

Each peak region represent the activity in a genomic region. These peaks are annotated for various features, including their genomic annotations and the results for differential accessibility analysis is also added there.

In [ ]:
rd_atac <- rowRanges(atac)

head(rd_atac)

Once we have the short sequencing reads, bioinformatics takes over to recreate the biological landscape.

* **Alignment:** We take every read and match it to a reference genome to find exactly where it originated.
* **Generating Coverage:** When thousands of reads map to the exact same open genomic location, they form a "pile-up." This is the black, mountainous signal we see in the example snapshot of chromatin accessibility.
* **Peak Calling:** We use statistical algorithms (like MACS2) to distinguish true biological pile-ups from random background noise, artifacts and duplicates. The algorithm defines the precise boundaries of these pile-ups, creating discrete peak regions (the shaded boxes on the example snapshot of chromatin accessibility) with specific coordinates (e.g., `chr1`, `start: 100`, `end: 400`).

## Statistical results

Now that we have genomic coordinates for our peaks, we can extract the specific information.

* **Genomic Annotation:** We compare our peak coordinates against a known database of genes. This tells us which gene is nearby (e.g., `Gene1`) and how far the peak is from the Transcription Start Site (`distanceTSS`).
* **Differential Accessibility:** To compare conditions, we count exactly how many sequencing reads fall within a specific peak region for each experimental group (e.g., `Group1` vs `Group2`). Statistical software (like DESeq2) then calculates the fold-change between these counts to determine if a region significantly opened or closed in response to a treatment. Here is the step-by-step bioinformatics pipeline used to generate those exact values from the raw data.

### 1. Creating a Consensus Peak Set

Before comparing samples, we need a unified map. The peak-calling algorithm (like MACS2) finds peaks for each sample individually. To compare them, bioinformatics tools merge all these individual peak files across all replicates and conditions into a single "master list" of non-overlapping genomic windows. This ensures we are comparing the exact same spatial coordinates across the board.

### 2. Generating the Count Matrix

Once the master list of regions is defined, we go back to the original aligned sequence alignments (the BAM files). Using tools like `featureCounts` or `bedtools`, we count exactly how many reads fall into each consensus region for every single sample. This generates a large matrix where the rows are the genomic regions and the columns are the individual samples.

### 3. Statistical Modeling

This raw count matrix is then fed into differential expression software, most commonly DESeq2 or edgeR.

* **Normalization:** The software mathematically adjusts for differences in sequencing depth between the samples.
* **Testing:** It fits a negative binomial model to test if the difference in read counts between groups (e.g., Group 1 vs. Group 2) is statistically significant.
* **Output:** The algorithm outputs a results table containing the $\text{Log}_{2} \text{ fold-change}$ (`logFC`), p-values, and adjusted p-values for every single region.

### 4. Constructing the `rowRanges`

Finally, the data is assembled for visualization tools like `EnrichedHeatmap`, which relies on the Bioconductor ecosystem. The genomic coordinates (`chr`, `start`, `end`) form the core of the `rowRanges` object, and the statistical results (like the `logFC` and p-values) are tacked on as metadata columns. This creates a unified object where the visualization tool instantly knows both *where* the region is located and *how significantly* its accessibility changed.

In [ ]:
# qvalue is the FDR adjusted P value
#rd_atac <- rd_atac[abs(rd_atac$logFC) >= 0.5 & rd_atac$qvalue <= 0.1]
rd_atac_filtered = subset(rd_atac, logFC >= .5 & qvalue <= .1)

# Add an "ATAC_" prefix to all the metadata columns in rd_atac_filtered
colnames(elementMetadata(rd_atac_filtered)) <- paste(
  "ATAC", colnames(elementMetadata(rd_atac_filtered )), sep = "_")

## Taking 1000 bp around the mid of ATAC-peaks

Most of the peaks are around 1000 bp wide. We can check that

In [ ]:
# median(lengths(rd_atac_filtered))  # `lengths` (plural) is an alias for `width`
median(width(rd_atac_filtered))

hist(
  width(rd_atac_filtered),
    breaks = 50,
    main = "Distribution of ATAC Peak Widths",
    xlab = "Width (bp)",
    col = "lightblue",
    border = "black")

For plotting the data, we can hence consider 1000 bp around the **peak-mid** that we find using `gr.mid()` (this returns the 1-bp center of each peak). It directly overwrites the data inside the `ranges` column.

In [ ]:
# Finding mid of all peaks
mid_peaks <- gr.mid(rd_atac_filtered)

# adding names to peaks to give them identity
names(mid_peaks) <- paste("peak", 1:length(mid_peaks), sep = "_")

There actually is a native way to do this in standard GenomicRanges, but it doesn't have a dedicated "midpoint" name. Instead, Bioconductor relies on its highly versatile `resize()` function.

In [ ]:
# Finding mid of all peaks
mid_peaks <- resize(rd_atac_filtered, width = 1, fix = "center")

# adding names to peaks to give them identity
names(mid_peaks) <- paste("peak", 1:length(mid_peaks), sep = "_")

A GRanges object is not a standard data frame. It strictly separates its data into two distinct parts: **Core Geometry** and **Metadata** . Because GRanges is an object-oriented structure (S4), you have to use specific "accessor" functions to extract the core geometry.

In [ ]:
# Returns an IRanges object with start, end, and width
ranges(mid_peaks)

_Note: Because GRanges is an S4 object, its core architecture is rigidly locked. You physically cannot add a custom geometry column (like midpeak) next to start and end. The rules of the object state that a GRanges row can only represent exactly one genomic interval at a time._

## Normalizing data for plotting

`bigWig` files are compressed, indexed, binary format used for efficiently displaying continuous data, like genomic signal data, in genome browsers. Here, we read in ATAC-seq `bigWig` files, filters the data to specific chromosomes, normalizes signal intensity around genomic regions of interest (peak centers), and saves the resulting matrices for downstream visualization.

In [ ]:
atac_files <- list.files("data", pattern = "ATAC", full.names = TRUE)
names(atac_files) <- gsub(pattern = "\\.bw", replacement = "", x = basename(atac_files))
atac_bw <- lapply(atac_files, function(x){
  a <- rtracklayer::import(x)
  a <- a[seqnames(a) %in% c("chr1", "chr2")]
  a
})

`bigWig` files are represented as `GRanges`.

In [ ]:
atac_bw

Next, we calculate the normalized signals into the area of our interest. Please check `?normalizeToMatrix` for details of this function

In [ ]:
mat_AS <- lapply(atac_bw, FUN = function(x) {
  normalizeToMatrix(x, mid_peaks,
    extend = 1000,
    value_column = "score",
    include_target = TRUE,
    mean_mode = "w0",
    w = 20,
    smooth = T,
    background = 0
  )
})

mat_AS

## Enriched heatmap

**Enriched heatmap** is a special type of heatmap which visualizes the enrichment of genomic signals over specific target regions.

In [ ]:
EnrichedHeatmap(mat = mat_AS$ATAC_11half, name = "E11.5") + 
  EnrichedHeatmap(mat = mat_AS$ATAC_15half, name = "E15.5")

Joining 2 `EnrichedHeatmaps` is very easy with a `+` sign.

## Changing aesthectics of Enriched heatmap

Let's work on one data for now.

### Changind color and size

In [ ]:
EnrichedHeatmap(
  mat = mat_AS$ATAC_11half,    # normalized matrix
  name = "E11.5",              # Name for the plot
  col = c("white", "red"),     # We change the colors for low to high values
  width = unit(4, "cm"),       # Width of the heatmap
  height = unit(8, "cm")      # Height of the heatmap
)

You may wonder why the color looks so light. The reason is in coverage values in ATAC, there exist some extreme values, which results in extreme value in `normalizedMatrix`.

### Color based on quantile

In [ ]:
# Taking data between 1 and 99 percentile
col_fun <- colorRamp2(quantile(mat_AS$ATAC_11half, c(0.01, 0.99)), c("white", "red"))

EnrichedHeatmap(
  mat = mat_AS$ATAC_11half,
  name = "E11.5",
  col = col_fun,
  width = unit(4, "cm"),
  height = unit(8, "cm")
)

### Changing some other aesthetics

In [ ]:
# We first change the color legent of the plot to show only 3 values
vmin <- as.numeric(quantile(mat_AS$ATAC_11half, c(0.01)))
vmax <- as.numeric(quantile(mat_AS$ATAC_11half, c(0.99)))
vmid <- (vmin + vmax) / 2
legend_ticks <- c(vmin, vmid, vmax)

EnrichedHeatmap(
  mat = mat_AS$ATAC_11half,
  name = "E11.5",
  col = col_fun,
  width = unit(4, "cm"),
  height = unit(8, "cm"),
  column_title = "E11.5",
  column_title_gp = gpar(fontsize = 10, fill = "#ffcccc"),
  axis_name = c("-1kb", "mid", "1kb"),  # We changed the axis names here
 heatmap_legend_param = list(
    at = legend_ticks,
    labels = round(legend_ticks, digits = 1),
    title_gp = gpar(fontsize = 8),
    labels_gp = gpar(fontsize = 7)
  ),
  top_annotation = HeatmapAnnotation(
    lines = anno_enriched(
      height = unit(2, "cm"),
      gp = gpar(
        lwd = 0.7,
        fontsize = 5
      ),
      axis_param = list(
        side = "right",
        facing = "inside",
        gp = gpar(
          fontsize = 7,
          col = "black",
          lwd = 0.4
        )
      )
    )
  )
)

### Split the Enriched heatmap based on `logFC` values

Although we see some signal here, but it might be a good idea to split the heatmap into the regions which gained and lost accessibility.

In [ ]:
split_change <- ifelse(mid_peaks$ATAC_logFC > 0, yes = "Increased accessibility", no = "Decreased accessibility")
names(split_change) <- names(mid_peaks)

head(split_change)

# Define cluster colors
cluster_colors <- c("Increased accessibility" = "red", "Decreased accessibility" = "blue")

# Make sure split_change has levels matching the color names
split_change <- factor(split_change, levels = names(cluster_colors))

EnrichedHeatmap(
  mat = mat_AS$ATAC_11half,
  name = "E11.5",
  row_split = split_change,
  col = col_fun,
  width = unit(4, "cm"),
  height = unit(8, "cm"),
  column_title = "E11.5",
  column_title_gp = gpar(fontsize = 10, fill = "#ffcccc"),
  axis_name = c("-1kb", "mid", "1kb"),
  heatmap_legend_param = list(
    at = legend_ticks,
    labels = round(legend_ticks, digits = 1),
    title_gp = gpar(fontsize = 8),
    labels_gp = gpar(fontsize = 7)
  ),
  top_annotation = HeatmapAnnotation(
    lines = anno_enriched(
      gp = gpar(col = cluster_colors),
      height = unit(2, "cm"),
      axis_param = list(
        side = "right",
        facing = "inside",
        gp = gpar(
          fontsize = 7,
          lwd = 0.4
        )
      )
    )
  )
)

### Make a function to make this heatmap

As you know we have at least 2 samples as of now. It will be a good idea to create a `function` to make this heatmap.

In [ ]:
make_EH <- function(norm_mat, heatmap_cols = c("white", "red"), split_rows = NULL, hm_name, col_fill = "#ffcccc"){
  col_fun <- colorRamp2(quantile(norm_mat, c(0.01, 0.99)), heatmap_cols)  
  
  vmin <- as.numeric(quantile(norm_mat, c(0.01)))
  vmax <- as.numeric(quantile(norm_mat, c(0.99)))
  vmid <- (vmin + vmax) / 2
  legend_ticks <- c(vmin, vmid, vmax)

EnrichedHeatmap(
  mat = norm_mat,
  name = hm_name,
  row_split = split_rows,
  col = col_fun,
  width = unit(4, "cm"),
  height = unit(8, "cm"),
  column_title = hm_name,
  column_title_gp = gpar(fontsize = 10, fill = col_fill),
  axis_name = c("-1kb", "mid", "1kb"),
  heatmap_legend_param = list(
    at = legend_ticks,
    labels = round(legend_ticks, digits = 1),
    title_gp = gpar(fontsize = 8),
    labels_gp = gpar(fontsize = 7)
  ),
  top_annotation = HeatmapAnnotation(
    lines = anno_enriched(
      gp = gpar(col = cluster_colors),
      height = unit(2, "cm"),
      axis_param = list(
        side = "right",
        facing = "inside",
        gp = gpar(
          fontsize = 7,
          lwd = 0.4
        )
      )
    )
  )
)
}

### Make Enriched Heatmaps for both ATAC samples

In [ ]:
eh_11h <- make_EH(norm_mat = mat_AS$ATAC_11half, split_rows = split_change, hm_name = "E11.5")
eh_15h <- make_EH(norm_mat = mat_AS$ATAC_15half, split_rows = split_change, hm_name = "E15.5", col_fill = "#e6fff2")

draw(eh_11h + eh_15h, merge_legend = TRUE)

## Another way to make annotations for split

It is probably a good idea to represent clusters with colors, instead of text

In [ ]:
eh_11h <- make_EH(norm_mat = mat_AS$ATAC_11half, hm_name = "E11.5")
eh_15h <- make_EH(norm_mat = mat_AS$ATAC_15half, hm_name = "E15.5", col_fill = "#e6fff2")

row_order_eh <- row_order(eh_11h)

anno_hm <- Heatmap(
  split_change,
  col = c("red", "blue"), 
  name = "Change",
  show_row_names = FALSE, 
  show_column_names = FALSE, 
  width = unit(3, "mm"),
  height = unit(8, "cm"),
  row_order = row_order_eh,
  row_title_gp = gpar(fontsize = 0)
)

draw(anno_hm + eh_11h + eh_15h, split = split_change, merge_legend = TRUE)

## Question 1

**Can you make split the Enriched Heatmap based on the annotations?**

**_Hint:_** `mid_peaks$ATAC_anno` contain annotations for the regions. 

:::{.callout-tip collapse="true"}

### Answer

In [ ]:
split_anno <- mid_peaks$ATAC_anno
names(split_anno) <- names(mid_peaks)

head(split_anno)

cols_an <- RColorBrewer::brewer.pal(n = length(unique(split_anno)), name = "Set1")

anno_an <- Heatmap(
  split_anno,
  col = cols_an, 
  name = "anno",
  show_row_names = FALSE, 
  show_column_names = FALSE, 
  width = unit(3, "mm"),
  height = unit(8, "cm"),
  row_order = row_order_eh,
  row_title_gp = gpar(fontsize = 0)
)

draw(anno_an + eh_11h + eh_15h, split = split_anno, merge_legend = TRUE)

:::

## Question 2

**Can you make split the Enriched Heatmap based on the annotations and change in direction?**

**_Hint:_** `mid_peaks$ATAC_anno` contain annotations for the regions. `mid_peaks$ATAC_logFC` contain sign of change.

:::{.callout-tip collapse="true"}

### Answer

In [ ]:
split_anno_dir <- paste(mid_peaks$ATAC_anno, ifelse(mid_peaks$ATAC_logFC > 0, yes = "Inc", no = "Dec"))
names(split_anno_dir) <- names(mid_peaks)

head(split_anno_dir)

cols_an <- RColorBrewer::brewer.pal(n = length(unique(split_anno_dir)), name = "Paired")

anno_an_dir <- Heatmap(
  split_anno_dir,
  col = cols_an, 
  name = "anno",
  show_row_names = FALSE, 
  show_column_names = FALSE, 
  width = unit(3, "mm"),
  height = unit(8, "cm"),
  row_order = row_order_eh,
  row_title_gp = gpar(fontsize = 0)
)

draw(anno_an_dir + eh_11h + eh_15h, split = split_anno_dir, merge_legend = TRUE)

:::

## Question 3

**Can you make split the Enriched Heatmap based on the annotations and change in direction with separate color bars for annotation and direction of change?**

**_Hint:_** `mid_peaks$ATAC_anno` contain annotations for the regions. `mid_peaks$ATAC_logFC` contain sign of change.

:::{.callout-tip collapse="true"}

### Answer

In [ ]:
split_anno_df <- data.frame(
  Annotation = mid_peaks$ATAC_anno,
  Direction = ifelse(mid_peaks$ATAC_logFC > 0, yes = "Inc", no = "Dec")
)

head(split_anno_df)

cols_an <- c("red", "blue", 
             RColorBrewer::brewer.pal(n = length(unique(split_anno_df$Annotation)), name = "Set1")
)

names(cols_an) <- c(unique(split_anno_df$Direction), unique(split_anno_df$Annotation))

anno_an_df <- Heatmap(
  split_anno_df,
  name = "anno",
  col = cols_an,
  show_row_names = FALSE, 
  show_column_names = FALSE, 
  width = unit(3, "mm"),
  height = unit(8, "cm"),
  row_order = row_order_eh,
  row_title_gp = gpar(fontsize = 0)
)

draw(anno_an_df + eh_11h + eh_15h, split = split_anno_dir, merge_legend = TRUE)

:::

:::{.callout-important}
**`Heatmap` and `EnrichedHeatmap` serve different functions, however the plots can be combined effortlessly. This makes  the visualization of complex data easy.**
:::